# LeetCode 221 — Maximal Square bằng Python

Link đề: https://leetcode.com/problems/maximal-square/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Matrix

Skeleton LeetCode:

```python
class Solution:
    def maximalSquare(self, matrix: List[List[str]]) -> int:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ bản mô tả gốc; mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với một lời giải brute force độc lập (thử mọi ô góc trên-trái và mọi cạnh, kiểm tra từng ô).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong ma trận chỉ gồm ký tự `'0'` và `'1'`, tìm **hình vuông lớn nhất chỉ chứa `'1'`** và trả về **diện tích** của nó (cạnh × cạnh).
2. Đi từ cách **vét cạn** (thử mọi góc và mọi cạnh) → **tổng tiền tố 2D + kiểm tra `O(1)`** (và chặt nhị phân theo cạnh) → **quy hoạch động 2D** `dp[i][j] = min(trên, trái, chéo) + 1` → **DP một hàng** (tiết kiệm bộ nhớ) → **histogram + stack đơn điệu**.
3. Hiểu **ý nghĩa của `dp[i][j]`** ("cạnh hình vuông lớn nhất có **góc dưới-phải** tại `(i, j)`") và **vì sao cần `min` của ba ô** (trên, trái, **chéo trên-trái**).
4. Làm quen với **ma trận 2D trong Python**: chỉ số `[i][j]`, ký tự `'1'` khác số `1`, cách tạo ma trận đúng (tránh lỗi tham chiếu chung hàng), và kỹ thuật **thêm hàng/cột đệm** để bỏ xử lý biên.
5. Tránh các lỗi kinh điển: trả về **cạnh** thay vì **diện tích**, so sánh `matrix[i][j] == 1` (sai kiểu), quên ô **chéo**, dùng `max` thay `min`, tạo `dp` bằng `[[0]*n]*m`.

## 2. Đề bài & ví dụ

Cho ma trận nhị phân `matrix` kích thước `m × n`, chỉ gồm các ký tự `'0'` và `'1'`. Tìm **hình vuông lớn nhất chỉ chứa `'1'`** và trả về **diện tích** của nó.

**Ví dụ 1**

- Input: `matrix = [["1","0","1","0","0"],["1","0","1","1","1"],["1","1","1","1","1"],["1","0","0","1","0"]]`
- Output: `4`
- Giải thích: hình vuông `2×2` lớn nhất toàn `1` (ví dụ ở hàng `1-2`, cột `2-3`) có diện tích `4`.

**Ví dụ 2**

- Input: `matrix = [["0","1"],["1","0"]]`
- Output: `1`
- Giải thích: chỉ có hình vuông `1×1`.

**Ví dụ 3**

- Input: `matrix = [["0"]]`
- Output: `0`
- Giải thích: không có ô `'1'` nào nên diện tích là `0`.

**Ràng buộc:** `m == matrix.length`; `n == matrix[i].length`; `1 <= m, n <= 300`; `matrix[i][j]` là `'0'` hoặc `'1'`.

In [1]:
from typing import List
import random
import time

examples = [
    ([["1", "0", "1", "0", "0"],
      ["1", "0", "1", "1", "1"],
      ["1", "1", "1", "1", "1"],
      ["1", "0", "0", "1", "0"]], 4),
    ([["0", "1"], ["1", "0"]], 1),
    ([["0"]], 0),
]
for matrix, expected in examples:
    print(f"matrix={matrix} -> kỳ vọng {expected}")

matrix=[['1', '0', '1', '0', '0'], ['1', '0', '1', '1', '1'], ['1', '1', '1', '1', '1'], ['1', '0', '0', '1', '0']] -> kỳ vọng 4
matrix=[['0', '1'], ['1', '0']] -> kỳ vọng 1
matrix=[['0']] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Ma trận là danh sách của danh sách

`matrix[i]` là **hàng** `i` (một danh sách); `matrix[i][j]` là ô ở **hàng `i`, cột `j`**. Số hàng `m = len(matrix)`, số cột `n = len(matrix[0])`. Chỉ số bắt đầu từ `0`.

In [2]:
matrix = [["1", "0", "1"],
          ["1", "1", "1"]]
m, n = len(matrix), len(matrix[0])
print(m, n)                     # 2 3
print(matrix[1], matrix[1][2])  # ['1', '1', '1'] 1
for i in range(m):
    for j in range(n):
        print(f"({i},{j})={matrix[i][j]}", end="  ")
    print()

2 3
['1', '1', '1'] 1
(0,0)=1  (0,1)=0  (0,2)=1  
(1,0)=1  (1,1)=1  (1,2)=1  


### 3.2. Ký tự `'1'` khác số `1`

Đề cho ma trận **ký tự**. So sánh `matrix[i][j] == 1` luôn là `False` vì `'1' == 1` là `False` trong Python. Phải so với chuỗi `'1'`, hoặc đổi sang số bằng `int(matrix[i][j])`.

In [3]:
print("1" == 1, "1" == "1", int("1") == 1)   # False True True
print(sum(int(ch) for ch in ["1", "0", "1"]))  # 2: đếm số '1' bằng cách đổi sang int

False True True
2


### 3.3. Tạo ma trận 2D đúng cách (và cạm bẫy `[[0] * n] * m`)

`[[0] * n for _ in range(m)]` tạo `m` hàng **độc lập**. Còn `[[0] * n] * m` tạo `m` **tham chiếu tới cùng một hàng**, sửa một ô thì cả cột đổi theo — lỗi cực kỳ hay gặp khi làm DP.

In [4]:
good = [[0] * 3 for _ in range(2)]
bad = [[0] * 3] * 2
good[0][0] = 7
bad[0][0] = 7
print("tạo đúng :", good)    # [[7, 0, 0], [0, 0, 0]]
print("tạo sai  :", bad)     # [[7, 0, 0], [7, 0, 0]]  (hai hàng là một!)

tạo đúng : [[7, 0, 0], [0, 0, 0]]
tạo sai  : [[7, 0, 0], [7, 0, 0]]


### 3.4. Hàng và cột đệm (padding) để khỏi xử lý biên

Công thức DP sẽ nhìn ô **trên**, **trái**, **chéo trên-trái**. Ở hàng/cột đầu tiên những ô đó không tồn tại; nếu dùng chỉ số `-1` Python sẽ **lấy phần tử cuối** (im lặng sai). Cách gọn: dùng `dp` kích thước `(m + 1) × (n + 1)` với hàng `0` và cột `0` toàn `0` (đệm). Ô `matrix[i][j]` ứng với `dp[i + 1][j + 1]`.

In [5]:
arr = [10, 20, 30]
print(arr[-1])        # 30, KHÔNG phải lỗi: nhưng nếu ta định "không có ô trái" thì đây là cái bẫy
m, n = 2, 3
dp = [[0] * (n + 1) for _ in range(m + 1)]
print(dp)             # hàng 0 và cột 0 là đệm toàn 0

30
[[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]]


### 3.5. Tổng tiền tố 2D (prefix sum 2D)

`P[i][j]` = tổng các số trong ma trận con từ `(0, 0)` đến `(i-1, j-1)`. Khi đó tổng ma trận con có góc trên-trái `(r, c)` và cạnh `s` là:

`P[r+s][c+s] - P[r][c+s] - P[r+s][c] + P[r][c]`.

Hình vuông cạnh `s` toàn `1` khi tổng đó bằng `s * s`. Ta dựng `P` một lần `O(mn)`, mỗi truy vấn `O(1)`.

In [6]:
grid = [[1, 0, 1], [1, 1, 1], [0, 1, 1]]
m, n = 3, 3
P = [[0] * (n + 1) for _ in range(m + 1)]
for i in range(m):
    for j in range(n):
        P[i + 1][j + 1] = grid[i][j] + P[i][j + 1] + P[i + 1][j] - P[i][j]
for row in P:
    print(row)


def square_sum(r, c, s):
    return P[r + s][c + s] - P[r][c + s] - P[r + s][c] + P[r][c]


print(square_sum(1, 1, 2), "== 2*2 ->", square_sum(1, 1, 2) == 4)    # hình vuông 2x2 ở góc (1,1): toàn 1
print(square_sum(0, 0, 2), "== 2*2 ->", square_sum(0, 0, 2) == 4)    # có số 0 nên tổng 3

[0, 0, 0, 0]
[0, 1, 1, 2]
[0, 2, 3, 5]
[0, 2, 4, 7]
4 == 2*2 -> True
3 == 2*2 -> False


### 3.6. Quy hoạch động (DP) là gì?

DP giải bài toán lớn bằng cách **lưu đáp án của bài toán con** rồi **dùng lại**. Ở đây bài toán con là: "hình vuông toàn `1` **lớn nhất có góc dưới-phải đúng tại ô `(i, j)`** có cạnh bao nhiêu?" Ta ghi đáp án vào `dp[i][j]`, và đáp án cuối cùng là **cạnh lớn nhất** trong mọi `dp[i][j]`, rồi bình phương.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét ví dụ 1:

```
1 0 1 0 0
1 0 1 1 1
1 1 1 1 1
1 0 0 1 0
```

Thử tìm hình vuông `3×3`: cần 9 ô `1` liền nhau, không có (cột `1` có `0` ở hàng `0,1,3`; ...). Thử `2×2`: hàng `1-2`, cột `2-3`: các ô `(1,2)=1`, `(1,3)=1`, `(2,2)=1`, `(2,3)=1` → đúng. Diện tích `4`.

**Cách nhìn quan trọng:** thay vì tìm hình vuông "ở đâu", hãy hỏi mỗi ô `(i, j)` **là góc dưới-phải** của hình vuông lớn nhất cạnh bao nhiêu. Nếu ô là `'0'` thì cạnh `0`. Nếu là `'1'`: hình vuông cạnh `s` kết thúc ở `(i, j)` tồn tại khi và chỉ khi **ba hình vuông cạnh `s-1`** kết thúc ở `(i-1, j)`, `(i, j-1)`, `(i-1, j-1)` đều tồn tại. Điều này dẫn tới công thức `min` ở phần 6.

## 5. Cách 1 — Brute Force / baseline: thử mọi góc trên-trái và mọi cạnh

**Ý tưởng.** Với mỗi ô `(r, c)` làm **góc trên-trái**, thử các cạnh `s = 1, 2, 3, ...` (không vượt biên). Với mỗi `s`, **kiểm tra từng ô** trong hình vuông `s × s`; nếu toàn `'1'` thì cập nhật đáp án, còn nếu gặp `'0'` thì **dừng tăng `s`** (mở rộng thêm chỉ làm hình vuông có thêm ô `0`).

In [7]:
def maximal_square_brute(matrix):
    m, n = len(matrix), len(matrix[0])
    best = 0
    for r in range(m):
        for c in range(n):
            if matrix[r][c] != "1":
                continue
            s = 1
            while r + s <= m and c + s <= n:
                # kiểm tra mọi ô của hình vuông s x s
                ok = all(matrix[i][j] == "1" for i in range(r, r + s) for j in range(c, c + s))
                if not ok:
                    break
                best = max(best, s)
                s += 1
    return best * best


for matrix, expected in examples:
    print(maximal_square_brute(matrix), "(kỳ vọng", expected, ")")

4 (kỳ vọng 4 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


**Độ phức tạp (Cách 1).** Gọi `m, n` là số hàng, cột, `L = min(m, n)`. Có `m·n` góc, mỗi góc thử tới `L` cạnh, mỗi cạnh kiểm tra `s²` ô: **Time `O(m·n·L³)`** trong worst case (với `300×300` là `~ 8·10^9`, **quá chậm**). **Space `O(1)`**.

**Nút thắt.** Mỗi lần tăng `s` ta **kiểm tra lại toàn bộ** hình vuông, trong khi nó chứa lại hình vuông `s-1` vừa kiểm tra. Hai hướng cải thiện: (a) kiểm tra hình vuông trong `O(1)` bằng **tổng tiền tố**; (b) **tái sử dụng** kết quả của hình vuông nhỏ hơn bằng **DP**.

## 6. Ý tưởng thuật toán / bước đột phá

**Công thức DP.** Gọi `dp[i][j]` là **cạnh** hình vuông toàn `1` lớn nhất có **góc dưới-phải** là `(i, j)`.

- Nếu `matrix[i][j] == '0'`: `dp[i][j] = 0`.
- Nếu `matrix[i][j] == '1'`: `dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])`.

**Vì sao `min` của ba ô (trên, trái, chéo)?** Hình vuông cạnh `s` có góc dưới-phải ở `(i, j)` gồm ô `(i, j)` và ba hình vuông cạnh `s - 1` kết thúc ở ô **trên** `(i-1, j)`, ô **trái** `(i, j-1)` và ô **chéo** `(i-1, j-1)`. Cả ba đều phải đạt cạnh **ít nhất `s - 1`**, nên `s - 1 <= min(ba giá trị)`; lớn nhất là `s = 1 + min`. Thiếu ô chéo, ta có thể chấp nhận một "lỗ" `0` ở chính giữa (xem phần lỗi).

**Đáp án** `= (max dp)^2`.

**Hai hướng khác:** (a) **Tổng tiền tố**: kiểm tra "hình vuông toàn `1`" bằng một phép tính `O(1)`; vì nếu có hình vuông cạnh `s` thì có cạnh `s-1` (cắt bớt), ta chặt **nhị phân theo `s`**. (b) **Histogram**: coi mỗi hàng là một biểu đồ cột (chiều cao = số `1` liên tiếp phía trên), hình vuông lớn nhất trong biểu đồ là `min(chiều cao, chiều rộng)` của các hình chữ nhật cực đại.

## 7. Các cách tiếp theo

### Cách 2 — Tổng tiền tố 2D + chặt nhị phân theo cạnh

**Giải thích.** Dựng `P` (tổng tiền tố của ma trận số `0/1`). Hàm `exists(s)` trả `True` nếu tồn tại hình vuông `s × s` toàn `1`: thử mọi góc trên-trái `(r, c)` và kiểm tra `square_sum == s*s` trong `O(1)`. Vì tính **đơn điệu** (có cạnh `s` thì có cạnh `s - 1`), chặt nhị phân `s` trong `[0, min(m, n)]`.

In [8]:
def build_prefix(matrix):
    m, n = len(matrix), len(matrix[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            P[i + 1][j + 1] = int(matrix[i][j]) + P[i][j + 1] + P[i + 1][j] - P[i][j]
    return P


def maximal_square_prefix_bsearch(matrix):
    m, n = len(matrix), len(matrix[0])
    P = build_prefix(matrix)

    def exists(s):
        for r in range(m - s + 1):
            for c in range(n - s + 1):
                if P[r + s][c + s] - P[r][c + s] - P[r + s][c] + P[r][c] == s * s:
                    return True
        return False

    lo, hi = 0, min(m, n)              # cạnh lớn nhất nằm trong [lo, hi]; lo luôn "tồn tại"
    while lo < hi:
        mid = (lo + hi + 1) // 2       # làm tròn lên để tránh vòng lặp vô hạn
        if exists(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo * lo


for matrix, expected in examples:
    print(maximal_square_prefix_bsearch(matrix), "(kỳ vọng", expected, ")")

4 (kỳ vọng 4 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


**Độ phức tạp Cách 2:** dựng `P` mất `O(mn)`; mỗi `exists(s)` tốn `O(mn)`; chặt nhị phân `O(log L)` lần với `L = min(m, n)`: **Time `O(m·n·log L)`**, **Space `O(m·n)`** cho `P`. Khoảng `300·300·9 ≈ 8·10^5` phép.

### Cách 3 — Quy hoạch động 2D (cách nộp kinh điển)

**Giải thích.** Dùng `dp` kích thước `(m+1) × (n+1)` có hàng/cột đệm `0`. Duyệt `i, j`: nếu ô là `'1'` thì `dp[i+1][j+1] = 1 + min(dp[i][j+1], dp[i+1][j], dp[i][j])` và cập nhật `best`; nếu `'0'` thì `dp = 0` (đã mặc định).

In [9]:
def maximal_square_dp2d(matrix):
    m, n = len(matrix), len(matrix[0])
    dp = [[0] * (n + 1) for _ in range(m + 1)]          # dp[i+1][j+1] ứng với ô (i, j); hàng/cột 0 là đệm
    best = 0
    for i in range(m):
        for j in range(n):
            if matrix[i][j] == "1":
                dp[i + 1][j + 1] = 1 + min(dp[i][j + 1],     # ô trên
                                           dp[i + 1][j],     # ô trái
                                           dp[i][j])         # ô chéo trên-trái
                best = max(best, dp[i + 1][j + 1])
    return best * best


for matrix, expected in examples:
    print(maximal_square_dp2d(matrix), "(kỳ vọng", expected, ")")

4 (kỳ vọng 4 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


In [10]:
# Code nộp LeetCode cho Cách 3
class Solution:
    def maximalSquare(self, matrix: List[List[str]]) -> int:
        m, n = len(matrix), len(matrix[0])
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        best = 0
        for i in range(m):
            for j in range(n):
                if matrix[i][j] == "1":
                    dp[i + 1][j + 1] = 1 + min(dp[i][j + 1], dp[i + 1][j], dp[i][j])
                    best = max(best, dp[i + 1][j + 1])
        return best * best


print(Solution().maximalSquare([["0", "1"], ["1", "1"]]))   # 1

1


**Độ phức tạp Cách 3:** **Time `O(m·n)`** (mỗi ô `O(1)`), **Space `O(m·n)`** cho bảng `dp`.

### Cách 4 — DP một hàng (tiết kiệm bộ nhớ)

**Giải thích.** `dp[i+1][*]` chỉ cần hàng `dp[i][*]` ngay trước nó. Giữ **một** mảng `row` độ dài `n + 1` (cột `0` là đệm). Khi cập nhật ô `(i, j)`, ta cần ba giá trị: `row[j+1]` (cũ, tức ô **trên**), `row[j]` (đã cập nhật, tức ô **trái**) và ô **chéo** là giá trị `row[j]` **trước** khi nó bị ghi đè ở bước trước — ta lưu vào biến `diag`.

In [11]:
def maximal_square_dp1row(matrix):
    m, n = len(matrix), len(matrix[0])
    row = [0] * (n + 1)              # row[j+1]: dp của ô (i, j) ở hàng hiện tại (sau khi cập nhật)
    best = 0
    for i in range(m):
        diag = 0                     # dp của ô chéo trên-trái = row[j] của hàng TRƯỚC (cột đệm: 0)
        for j in range(n):
            up = row[j + 1]          # ô trên: giá trị cũ của hàng trước
            left = row[j]            # ô trái: giá trị mới của hàng hiện tại
            if matrix[i][j] == "1":
                row[j + 1] = 1 + min(up, left, diag)
                best = max(best, row[j + 1])
            else:
                row[j + 1] = 0
            diag = up                # sang cột kế: ô chéo của (i, j+1) chính là ô trên của (i, j)
    return best * best


for matrix, expected in examples:
    print(maximal_square_dp1row(matrix), "(kỳ vọng", expected, ")")

4 (kỳ vọng 4 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


**Độ phức tạp Cách 4:** **Time `O(m·n)`**, **Space `O(n)`** (một mảng `row`; có thể chọn chiều ngắn hơn làm `n` để còn `O(min(m, n))`).

### Cách 5 — Histogram + stack đơn điệu

**Giải thích.** Với mỗi hàng `i`, `heights[j]` = số `'1'` **liên tiếp** kết thúc ở hàng `i` tại cột `j` (nếu ô là `'0'` thì `0`). Hình vuông lớn nhất có cạnh dưới nằm ở hàng `i` chính là hình vuông lớn nhất trong **biểu đồ cột** `heights`. Dùng **stack đơn điệu tăng** để tìm mọi hình chữ nhật cực đại `(cao h, rộng w)`; hình vuông lớn nhất bên trong nó có cạnh `min(h, w)`.

In [12]:
def maximal_square_histogram(matrix):
    m, n = len(matrix), len(matrix[0])
    heights = [0] * n
    best = 0
    for i in range(m):
        for j in range(n):
            heights[j] = heights[j] + 1 if matrix[i][j] == "1" else 0
        stack = []                                     # chứa chỉ số cột, chiều cao tăng dần
        for j in range(n + 1):
            h = heights[j] if j < n else 0             # cột "lính canh" chiều cao 0 ở cuối để xả hết stack
            while stack and heights[stack[-1]] >= h:
                height = heights[stack.pop()]
                left = stack[-1] if stack else -1      # biên trái (cột nhỏ hơn gần nhất)
                width = j - left - 1                   # chiều rộng của hình chữ nhật cực đại
                best = max(best, min(height, width))
            stack.append(j)
    return best * best


for matrix, expected in examples:
    print(maximal_square_histogram(matrix), "(kỳ vọng", expected, ")")

4 (kỳ vọng 4 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


**Độ phức tạp Cách 5:** **Time `O(m·n)`** (mỗi hàng, mỗi cột vào/ra stack tối đa một lần), **Space `O(n)`**. Cách này là phiên bản "ít người nghĩ ra" nhưng liên hệ trực tiếp với bài **Maximal Rectangle** (LeetCode 85).

> Cách 3 (DP 2D) hoặc Cách 4 (DP một hàng) là cách nộp phổ biến; Cách 4 nên nhớ vì cùng kỹ thuật "mảng lăn" dùng cho nhiều DP khác.

## 8. TRACE từng bước

Ta in `dp` sau mỗi hàng của Cách 3 trên ví dụ 1, kèm quyết định ở vài ô quan trọng.

In [13]:
def trace_dp2d(matrix):
    m, n = len(matrix), len(matrix[0])
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    best = 0
    print("matrix:")
    for row in matrix:
        print("  ", " ".join(row))
    for i in range(m):
        for j in range(n):
            if matrix[i][j] == "1":
                up, left, diag = dp[i][j + 1], dp[i + 1][j], dp[i][j]
                dp[i + 1][j + 1] = 1 + min(up, left, diag)
                best = max(best, dp[i + 1][j + 1])
                if i >= 1 and j >= 1:
                    print(f"ô ({i},{j})='1': trên={up}, trái={left}, chéo={diag} -> dp = 1 + min = {dp[i + 1][j + 1]}")
        print(f"dp sau hàng {i}: ", dp[i + 1][1:], " (best =", best, ")")
    print("cạnh lớn nhất =", best, "-> diện tích =", best * best)
    return best * best


trace_dp2d(examples[0][0])

matrix:
   1 0 1 0 0
   1 0 1 1 1
   1 1 1 1 1
   1 0 0 1 0
dp sau hàng 0:  [1, 0, 1, 0, 0]  (best = 1 )
ô (1,2)='1': trên=1, trái=0, chéo=0 -> dp = 1 + min = 1
ô (1,3)='1': trên=0, trái=1, chéo=1 -> dp = 1 + min = 1
ô (1,4)='1': trên=0, trái=1, chéo=0 -> dp = 1 + min = 1
dp sau hàng 1:  [1, 0, 1, 1, 1]  (best = 1 )
ô (2,1)='1': trên=0, trái=1, chéo=1 -> dp = 1 + min = 1
ô (2,2)='1': trên=1, trái=1, chéo=0 -> dp = 1 + min = 1
ô (2,3)='1': trên=1, trái=1, chéo=1 -> dp = 1 + min = 2
ô (2,4)='1': trên=1, trái=2, chéo=1 -> dp = 1 + min = 2
dp sau hàng 2:  [1, 1, 1, 2, 2]  (best = 2 )
ô (3,3)='1': trên=2, trái=0, chéo=1 -> dp = 1 + min = 1
dp sau hàng 3:  [1, 0, 0, 1, 0]  (best = 2 )
cạnh lớn nhất = 2 -> diện tích = 4


4

In [14]:
def trace_dp1row(matrix):
    m, n = len(matrix), len(matrix[0])
    row = [0] * (n + 1)
    best = 0
    for i in range(m):
        diag = 0
        print(f"--- hàng {i}: row bắt đầu = {row[1:]}")
        for j in range(n):
            up, left = row[j + 1], row[j]
            row[j + 1] = 1 + min(up, left, diag) if matrix[i][j] == "1" else 0
            best = max(best, row[j + 1])
            print(f"  j={j}: ô={matrix[i][j]} up={up} left={left} diag={diag} -> row={row[1:]}")
            diag = up
    return best * best


trace_dp1row([["1", "1", "1"], ["1", "1", "1"], ["1", "1", "0"]])

--- hàng 0: row bắt đầu = [0, 0, 0]
  j=0: ô=1 up=0 left=0 diag=0 -> row=[1, 0, 0]
  j=1: ô=1 up=0 left=1 diag=0 -> row=[1, 1, 0]
  j=2: ô=1 up=0 left=1 diag=0 -> row=[1, 1, 1]
--- hàng 1: row bắt đầu = [1, 1, 1]
  j=0: ô=1 up=1 left=0 diag=0 -> row=[1, 1, 1]
  j=1: ô=1 up=1 left=1 diag=1 -> row=[1, 2, 1]
  j=2: ô=1 up=1 left=2 diag=1 -> row=[1, 2, 2]
--- hàng 2: row bắt đầu = [1, 2, 2]
  j=0: ô=1 up=1 left=0 diag=0 -> row=[1, 2, 2]
  j=1: ô=1 up=2 left=1 diag=1 -> row=[1, 2, 2]
  j=2: ô=0 up=2 left=2 diag=2 -> row=[1, 2, 0]


4

Bảng `dp` (ví dụ 1; cột 0 là đệm không in):

| hàng | `dp[i][0..4]` |
|---|---|
| 0 | `[1, 0, 1, 0, 0]` |
| 1 | `[1, 0, 1, 1, 1]` |
| 2 | `[1, 1, 1, 2, 2]` |
| 3 | `[1, 0, 0, 1, 0]` |

Giá trị lớn nhất là `2` (tại `(2,3)` và `(2,4)`): ô `(2,3)` có trên `1`, trái `1`, chéo `1` ⇒ `1 + 1 = 2`. Đáp án `2² = 4`.

## 9. Code nộp LeetCode

Bản nộp dùng DP một hàng (Cách 4) cho bộ nhớ `O(n)`; DP 2D (Cách 3) cũng hoàn toàn được. Đúng `class Solution`, đúng `maximalSquare(self, matrix: List[List[str]]) -> int`.

In [15]:
class Solution:
    def maximalSquare(self, matrix: List[List[str]]) -> int:
        m, n = len(matrix), len(matrix[0])
        row = [0] * (n + 1)                  # row[j + 1] = cạnh lớn nhất có góc dưới-phải ở cột j của hàng hiện tại
        best = 0
        for i in range(m):
            diag = 0                         # dp ô chéo trên-trái (cột đệm = 0)
            for j in range(n):
                up = row[j + 1]
                if matrix[i][j] == "1":
                    row[j + 1] = 1 + min(up, row[j], diag)
                    best = max(best, row[j + 1])
                else:
                    row[j + 1] = 0
                diag = up
        return best * best


sol = Solution()
for matrix, expected in examples:
    assert sol.maximalSquare(matrix) == expected
print("OK: 3 ví dụ của đề")

OK: 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Định nghĩa `dp`:** `dp[i][j]` là cạnh lớn nhất của hình vuông toàn `1` có **góc dưới-phải** `(i, j)`. Mọi hình vuông toàn `1` đều có một góc dưới-phải duy nhất, nên **cạnh lớn nhất toàn ma trận = max của `dp`**.
- **Cận trên:** nếu có hình vuông cạnh `s` kết thúc ở `(i, j)` thì ba ô trên/trái/chéo mỗi ô là góc dưới-phải của hình vuông cạnh `s - 1` (cắt bớt hàng/cột cuối) nên mỗi `dp` đó `>= s - 1`, tức `s <= 1 + min`.
- **Cận dưới:** nếu ba ô trên/trái/chéo đều có hình vuông cạnh `t = min`, ghép với ô `(i, j)` (là `'1'`) ta được hình vuông cạnh `t + 1`: ba hình vuông cạnh `t` phủ cùng nhau vùng `(t+1) × (t+1)` trừ chính ô `(i, j)`; ô này là `'1'`. Vậy `dp[i][j] >= 1 + min`. Hai chiều cho `dp = 1 + min`.
- **Biên:** hàng/cột đệm bằng `0` cho `dp = 1` ở ô đầu hàng/cột mà là `'1'` (hình vuông `1×1`).
- **DP một hàng:** `up = row[j + 1]` (giá trị cũ), `left = row[j]` (giá trị mới ở cột trái), `diag` (giá trị cũ của `row[j]` trước khi ghi đè) — cùng ba giá trị như bảng 2D.
- **Prefix + nhị phân:** tính đơn điệu (cạnh `s` tồn tại ⇒ cạnh `s - 1` tồn tại) làm chặt nhị phân hợp lệ; tổng `= s²` ⇔ toàn `1`.

## 11. Phân tích Time / Space Complexity

Gọi `m, n` là số hàng, cột; `L = min(m, n)`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(m·n·L³)` | `O(1)` |
| 2. Tổng tiền tố + nhị phân | `O(m·n·log L)` | `O(m·n)` |
| 3. DP 2D | `O(m·n)` | `O(m·n)` |
| 4. DP một hàng | `O(m·n)` | `O(n)` |
| 5. Histogram + stack | `O(m·n)` | `O(n)` |

Với `m = n = 300`: `m·n = 90 000` — cực nhỏ; vét cạn (Cách 1) có thể tới hàng tỉ phép nên **không dùng cho input lớn**. Không có khác biệt "average/worst" đáng kể ở các cách DP (luôn duyệt đủ `m·n` ô); Cách 1 thì worst case là ma trận toàn `1`.

## 12. Bộ test

`oracle` là brute force độc lập: với **mọi** góc `(r, c)` và **mọi** cạnh `s` (không dùng `break`), kiểm tra bằng `all(...)` trên từng ô. Ta đối chiếu mọi cách với nó.

In [16]:
def oracle(matrix):
    """Brute force độc lập: thử mọi (r, c, s), kiểm tra từng ô của hình vuông."""
    m, n = len(matrix), len(matrix[0])
    best = 0
    for r in range(m):
        for c in range(n):
            for s in range(1, min(m - r, n - c) + 1):
                cells = [matrix[i][j] for i in range(r, r + s) for j in range(c, c + s)]
                if all(ch == "1" for ch in cells):
                    best = max(best, s)
    return best * best


impls = {
    "brute": maximal_square_brute,
    "prefix_bsearch": maximal_square_prefix_bsearch,
    "dp2d": maximal_square_dp2d,
    "dp1row": maximal_square_dp1row,
    "histogram": maximal_square_histogram,
    "submit": sol.maximalSquare,
}

for matrix, expected in examples:
    assert oracle(matrix) == expected
    for key, fn in impls.items():
        assert fn(matrix) == expected, (key, matrix, fn(matrix), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên

In [17]:
def grid(*rows):
    return [list(r) for r in rows]


edge_cases = [
    (grid("1"), 1, "một ô '1'"),
    (grid("0"), 0, "một ô '0'"),
    (grid("1111"), 1, "một hàng toàn 1: chỉ vuông 1x1"),
    (grid("1", "1", "1"), 1, "một cột toàn 1"),
    (grid("11", "11"), 4, "2x2 toàn 1"),
    (grid("111", "111", "111"), 9, "3x3 toàn 1"),
    (grid("111", "101", "111"), 1, "vòng 1 có lỗ ở giữa: không có vuông 2x2"),
    (grid("01", "11"), 1, "ca thiếu ô chéo: trên=1, trái=1 nhưng chéo=0"),
    (grid("0000", "0000"), 0, "toàn 0"),
    (grid("11110", "11110", "11110", "00001"), 9, "vuông 3x3 trong ma trận chữ nhật"),
    (grid("10100", "10111", "11111", "10010"), 4, "ví dụ 1 viết dạng chuỗi"),
]
for matrix, expected, label in edge_cases:
    assert oracle(matrix) == expected, (label, oracle(matrix), expected)
    for key, fn in impls.items():
        assert fn(matrix) == expected, (key, label, fn(matrix), expected)
    print("PASS", label)

PASS một ô '1'
PASS một ô '0'
PASS một hàng toàn 1: chỉ vuông 1x1
PASS một cột toàn 1
PASS 2x2 toàn 1
PASS 3x3 toàn 1
PASS vòng 1 có lỗ ở giữa: không có vuông 2x2
PASS ca thiếu ô chéo: trên=1, trái=1 nhưng chéo=0
PASS toàn 0
PASS vuông 3x3 trong ma trận chữ nhật
PASS ví dụ 1 viết dạng chuỗi


### 12.2. Tính chất và phản ví dụ

- Đáp án luôn là **số chính phương** (`0, 1, 4, 9, ...`) và `<= (min(m, n))²`; bằng `0` khi và chỉ khi không có ô `'1'`.
- **Chuyển vị** ma trận (đổi hàng ↔ cột) và **lật** ngang/dọc **không đổi** đáp án.
- **Phản ví dụ** cho lỗi thiếu ô chéo: `[["0","1"],["1","1"]]` có đáp án `1` nhưng công thức `1 + min(trên, trái)` cho `4`.

In [18]:
random.seed(221)
for _ in range(300):
    m, n = random.randint(1, 7), random.randint(1, 7)
    mat = [[random.choice("01") for _ in range(n)] for _ in range(m)]
    ans = sol.maximalSquare(mat)
    side = int(round(ans ** 0.5))
    assert side * side == ans and side <= min(m, n)
    assert (ans == 0) == all(ch == "0" for row in mat for ch in row)
    transposed = [list(col) for col in zip(*mat)]
    assert sol.maximalSquare(transposed) == ans
    assert sol.maximalSquare([r[::-1] for r in mat]) == ans
    assert sol.maximalSquare(mat[::-1]) == ans
print("PASS tính chất (chính phương, chuyển vị, lật) trên 300 ma trận")


def sai_thieu_cheo(matrix):
    m, n = len(matrix), len(matrix[0])
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    best = 0
    for i in range(m):
        for j in range(n):
            if matrix[i][j] == "1":
                dp[i + 1][j + 1] = 1 + min(dp[i][j + 1], dp[i + 1][j])      # SAI: thiếu ô chéo
                best = max(best, dp[i + 1][j + 1])
    return best * best


bad = [["0", "1"], ["1", "1"]]
print("phản ví dụ:", bad, "-> đúng", sol.maximalSquare(bad), "| thiếu chéo:", sai_thieu_cheo(bad))
assert sai_thieu_cheo(bad) != sol.maximalSquare(bad)

PASS tính chất (chính phương, chuyển vị, lật) trên 300 ma trận
phản ví dụ: [['0', '1'], ['1', '1']] -> đúng 1 | thiếu chéo: 4


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [19]:
random.seed(4221)
for trial in range(1500):
    m, n = random.randint(1, 9), random.randint(1, 9)
    p = random.choice([0.3, 0.6, 0.85, 0.97])
    mat = [["1" if random.random() < p else "0" for _ in range(n)] for _ in range(m)]
    truth = oracle(mat)
    for key, fn in impls.items():
        assert fn(mat) == truth, (key, mat, fn(mat), truth)
print("PASS 1500 ma trận nhỏ (mật độ '1' từ thưa đến dày): mọi cách khớp oracle")

# Ma trận lớn hơn: bỏ oracle/brute (quá chậm), so các cách nhanh với nhau
fast = {k: v for k, v in impls.items() if k not in ("brute",)}
for trial in range(60):
    m, n = random.randint(20, 60), random.randint(20, 60)
    p = random.choice([0.5, 0.9, 0.98])
    mat = [["1" if random.random() < p else "0" for _ in range(n)] for _ in range(m)]
    ref = maximal_square_dp2d(mat)
    for key, fn in fast.items():
        assert fn(mat) == ref, (key, m, n, fn(mat), ref)
print("PASS 60 ma trận cỡ 20..60: các cách nhanh khớp nhau (DP 2D làm chuẩn, đã kiểm chứng với oracle ở trên)")

PASS 1500 ma trận nhỏ (mật độ '1' từ thưa đến dày): mọi cách khớp oracle


PASS 60 ma trận cỡ 20..60: các cách nhanh khớp nhau (DP 2D làm chuẩn, đã kiểm chứng với oracle ở trên)


### 12.4. Đo thời gian trên ma trận lớn (tham khảo)

Ma trận `300 × 300` (cỡ tối đa), mật độ `'1'` cao (`98%`) để có hình vuông lớn.

In [20]:
random.seed(300)
big = [["1" if random.random() < 0.98 else "0" for _ in range(300)] for _ in range(300)]
for key in ("prefix_bsearch", "dp2d", "dp1row", "histogram"):
    t0 = time.perf_counter()
    ans = impls[key](big)
    dt = (time.perf_counter() - t0) * 1000
    print(f"{key:<16} ans={ans:>6}  {dt:8.1f} ms")
all_ones = [["1"] * 300 for _ in range(300)]
t0 = time.perf_counter()
print("toàn '1' 300x300 (DP 1 hàng):", sol.maximalSquare(all_ones), f"{(time.perf_counter() - t0) * 1000:.1f} ms")

prefix_bsearch   ans=   484      59.3 ms
dp2d             ans=   484      14.6 ms
dp1row           ans=   484      11.3 ms


histogram        ans=   484      20.4 ms
toàn '1' 300x300 (DP 1 hàng): 90000 11.4 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Trả về cạnh thay vì diện tích
#### Code sai

In [21]:
def sai_tra_canh(matrix):
    m, n = len(matrix), len(matrix[0])
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    best = 0
    for i in range(m):
        for j in range(n):
            if matrix[i][j] == "1":
                dp[i + 1][j + 1] = 1 + min(dp[i][j + 1], dp[i + 1][j], dp[i][j])
                best = max(best, dp[i + 1][j + 1])
    return best                                              # SAI: đề yêu cầu diện tích


print("đúng:", Solution().maximalSquare(examples[0][0]), "| sai:", sai_tra_canh(examples[0][0]))

đúng: 4 | sai: 2


**Vì sao sai:** `dp` lưu **cạnh**; đề hỏi **diện tích** `= cạnh²`. **Code đúng:** `return best * best`.

### Lỗi 2: So sánh `matrix[i][j] == 1` (số) thay vì `'1'` (ký tự)
#### Code sai

In [22]:
def sai_so_sanh_kieu(matrix):
    m, n = len(matrix), len(matrix[0])
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    best = 0
    for i in range(m):
        for j in range(n):
            if matrix[i][j] == 1:                            # SAI: ma trận chứa chuỗi '1'
                dp[i + 1][j + 1] = 1 + min(dp[i][j + 1], dp[i + 1][j], dp[i][j])
                best = max(best, dp[i + 1][j + 1])
    return best * best


print("đúng:", Solution().maximalSquare(examples[0][0]), "| sai:", sai_so_sanh_kieu(examples[0][0]))

đúng: 4 | sai: 0


**Vì sao sai:** `'1' == 1` là `False`, nên điều kiện không bao giờ đúng và luôn trả `0`. **Code đúng:** `matrix[i][j] == "1"` hoặc `int(matrix[i][j]) == 1`.

### Lỗi 3: Thiếu ô chéo (chỉ lấy `min(trên, trái)`)
Xem `sai_thieu_cheo` ở phần test: ma trận `[["0","1"],["1","1"]]` có trên `1`, trái `1` nhưng chéo `0`, nên hình vuông `2×2` **không tồn tại**; thiếu chéo cho kết quả `4`. **Code đúng:** `min` của **ba** ô.

### Lỗi 4: Dùng `max` thay vì `min`
#### Code sai

In [23]:
def sai_dung_max(matrix):
    m, n = len(matrix), len(matrix[0])
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    best = 0
    for i in range(m):
        for j in range(n):
            if matrix[i][j] == "1":
                dp[i + 1][j + 1] = 1 + max(dp[i][j + 1], dp[i + 1][j], dp[i][j])    # SAI: max
                best = max(best, dp[i + 1][j + 1])
    return best * best


print("đúng:", Solution().maximalSquare(examples[0][0]), "| sai:", sai_dung_max(examples[0][0]))

đúng: 4 | sai: 49


**Vì sao sai:** hình vuông cần **cả ba** hướng đều đủ lớn, nên bị chặn bởi hướng **yếu nhất** (`min`). `max` đếm quá tay.

### Lỗi 5: Tạo `dp` bằng `[[0] * n] * m`
#### Code sai

In [24]:
def sai_tao_dp(matrix):
    m, n = len(matrix), len(matrix[0])
    dp = [[0] * (n + 1)] * (m + 1)                           # SAI: m+1 hàng cùng một đối tượng
    best = 0
    for i in range(m):
        for j in range(n):
            if matrix[i][j] == "1":
                dp[i + 1][j + 1] = 1 + min(dp[i][j + 1], dp[i + 1][j], dp[i][j])
                best = max(best, dp[i + 1][j + 1])
    return best * best


print("đúng:", Solution().maximalSquare(examples[0][0]), "| sai:", sai_tao_dp(examples[0][0]))

đúng: 4 | sai: 9


**Vì sao sai:** mọi hàng của `dp` là **cùng một danh sách**; ghi `dp[i+1][j+1]` làm đổi ở **mọi hàng**, dữ liệu bị lẫn. **Code đúng:** `[[0] * (n + 1) for _ in range(m + 1)]`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**hình vuông / hình chữ nhật lớn nhất** trong ma trận" → nghĩ **DP theo góc dưới-phải** (hình vuông) hoặc **histogram + stack** (hình chữ nhật).
- Công thức DP hình vuông: **`dp = 1 + min(trên, trái, chéo)`** nếu ô hợp lệ, `0` nếu không. Nhớ đủ **ba** ô.
- **Đệm** một hàng và một cột `0` để bỏ xử lý biên; nhớ dịch chỉ số (`dp[i+1][j+1]`).
- DP chỉ nhìn **hàng trước** → rút gọn thành **mảng một hàng** + biến `diag` giữ ô chéo.
- Đáp án là **diện tích** → bình phương cạnh lớn nhất.
- Có tính **đơn điệu** (có cạnh `s` thì có `s-1`) → nghĩ **chặt nhị phân** + kiểm tra bằng tổng tiền tố.
- Luôn thử: toàn `0`, toàn `1`, một hàng/một cột, vòng `1` có lỗ ở giữa.

## 15. Biến thể của bài

1. **Maximal Rectangle** (LeetCode 85): hình chữ nhật lớn nhất toàn `1` — histogram + stack (Cách 5 mở rộng).
2. **Count Square Submatrices with All Ones** (1277): đếm mọi hình vuông toàn `1` — cộng **tất cả** `dp[i][j]`.
3. **Largest Plus Sign** (764), **Largest 1-Bordered Square** (1139): biến thể DP đếm 4 hướng.
4. **Matrix Block Sum** (1314), **Range Sum Query 2D** (304): tổng tiền tố 2D.
5. Trả về cả **vị trí** (góc trên-trái) của hình vuông lớn nhất.

In [25]:
def count_squares(matrix):
    """LeetCode 1277: tổng mọi dp[i][j] = số hình vuông toàn 1 (mỗi ô là góc dưới-phải của dp[i][j] hình vuông)."""
    m, n = len(matrix), len(matrix[0])
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    total = 0
    for i in range(m):
        for j in range(n):
            if matrix[i][j] == "1":
                dp[i + 1][j + 1] = 1 + min(dp[i][j + 1], dp[i + 1][j], dp[i][j])
                total += dp[i + 1][j + 1]
    return total


def largest_square_with_position(matrix):
    m, n = len(matrix), len(matrix[0])
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    best, pos = 0, None
    for i in range(m):
        for j in range(n):
            if matrix[i][j] == "1":
                dp[i + 1][j + 1] = 1 + min(dp[i][j + 1], dp[i + 1][j], dp[i][j])
                if dp[i + 1][j + 1] > best:
                    best = dp[i + 1][j + 1]
                    pos = (i - best + 1, j - best + 1)       # góc trên-trái
    return best, pos


print(count_squares([["0", "1", "1", "1"], ["1", "1", "1", "1"], ["0", "1", "1", "1"]]))   # 15
print(largest_square_with_position(examples[0][0]))                                        # (2, (1, 2)): cạnh 2, góc trên-trái (1, 2)

15
(2, (1, 2))


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi góc và mọi cạnh, kiểm tra từng ô | `O(m·n·L³)` | `O(1)` | làm oracle kiểm tra, `m, n` rất nhỏ |
| 2. Prefix 2D + nhị phân | kiểm tra "toàn 1" trong `O(1)`, chặt nhị phân cạnh | `O(m·n·log L)` | `O(m·n)` | khi cần nhiều truy vấn tổng 2D |
| 3. DP 2D | `dp = 1 + min(trên, trái, chéo)` | `O(m·n)` | `O(m·n)` | **cách kinh điển**, dễ giải thích |
| 4. DP một hàng | mảng lăn + biến `diag` | `O(m·n)` | `O(n)` | **bản nộp tiết kiệm bộ nhớ** |
| 5. Histogram + stack | hình chữ nhật cực đại theo hàng, lấy `min(cao, rộng)` | `O(m·n)` | `O(n)` | liên hệ Maximal Rectangle |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `"1" == 1`; (b) `[[0] * 2] * 2` sau khi gán `a[0][0] = 5` cho `a = [[0] * 2] * 2`; (c) `min(2, 3, 2) + 1`; (d) `Solution().maximalSquare([["1","1"],["1","1"]])`.

**Bài 2 (tính bằng tay).** Tính bảng `dp` cho ma trận `["111", "111", "101"]` (mỗi chuỗi là một hàng) và cho biết đáp án.

**Bài 3 (điền TODO).** Hoàn thiện `maximal_square_todo(matrix)` (DP 2D có đệm).

In [26]:
def maximal_square_todo(matrix):
    m, n = len(matrix), len(matrix[0])
    # TODO 1: tạo dp kích thước (m + 1) x (n + 1) toàn 0 bằng list comprehension
    best = 0
    # TODO 2: duyệt i trong range(m), j trong range(n)
    # TODO 3: nếu matrix[i][j] == "1": dp[i + 1][j + 1] = 1 + min(dp[i][j + 1], dp[i + 1][j], dp[i][j])
    # TODO 4: cập nhật best = max(best, dp[i + 1][j + 1])
    # TODO 5: trả best * best
    pass


# Tự làm xong rồi mới mở test:
# for matrix, expected in examples:
#     assert maximal_square_todo(matrix) == expected

**Bài 4 (tự viết lại).** Viết `maximal_square_rows(matrix)` bằng DP **một hàng** (mảng `row` và biến `diag`), **không nhìn lại** Cách 4.

**Bài 5 (biến thể / mini challenge).** Viết `count_squares2(matrix)` đếm **tổng số** hình vuông con toàn `1` (mọi kích thước) bằng cách cộng `dp`. Thử với `["0111", "1111", "0111"]` → `15`. Giải thích bằng lời vì sao một ô có `dp = 3` đóng góp đúng `3` hình vuông (cạnh `1`, `2`, `3`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: kiểu dữ liệu `str` khác `int`; `[[0]*2]*2` là hai tham chiếu tới cùng một hàng. Bài 2: tính theo từng ô, dùng công thức `1 + min(trên, trái, chéo)` và hàng/cột đệm bằng 0. Bài 5: số hình vuông có góc dưới-phải tại `(i, j)` bằng `dp[i][j]`.
- **Mức 2 (vừa):** Bài 1: `False`, `[[5, 0], [5, 0]]`, `3`, `4`. Bài 2: hàng 0: `1 1 1`; hàng 1: `1 2 2`; hàng 2: `1 0 ...` (ô `(2,1)` là `'0'`, nên `0`); ô `(2,2)` là `'1'`: trên `2`, trái `0`, chéo `2` ⇒ `1`. Bài 3: nhớ `dp[i + 1][j + 1]`. Bài 4: `diag = up` ở cuối mỗi bước `j`.
- **Mức 3 (sâu):** Bài 2: `dp` (không in đệm) là `[[1,1,1],[1,2,2],[1,0,1]]`, cạnh lớn nhất `2` ⇒ đáp án `4`. Bài 4: xem Cách 4 (khởi tạo `diag = 0` mỗi hàng). Bài 5: gọi `total += dp[i + 1][j + 1]` mỗi lần ô là `'1'`; ô có `dp = 3` là góc dưới-phải của các hình vuông cạnh `1`, `2`, `3`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert ("1" == 1) is False
a = [[0] * 2] * 2
a[0][0] = 5
assert a == [[5, 0], [5, 0]]
assert min(2, 3, 2) + 1 == 3
assert Solution().maximalSquare([["1", "1"], ["1", "1"]]) == 4

# Bài 2
mat2 = grid("111", "111", "101")
assert oracle(mat2) == 4 and Solution().maximalSquare(mat2) == 4


# Bài 3
def maximal_square_todo_ans(matrix):
    m, n = len(matrix), len(matrix[0])
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    best = 0
    for i in range(m):
        for j in range(n):
            if matrix[i][j] == "1":
                dp[i + 1][j + 1] = 1 + min(dp[i][j + 1], dp[i + 1][j], dp[i][j])
                best = max(best, dp[i + 1][j + 1])
    return best * best


for matrix, expected in examples:
    assert maximal_square_todo_ans(matrix) == expected


# Bài 4
def maximal_square_rows(matrix):
    m, n = len(matrix), len(matrix[0])
    row = [0] * (n + 1)
    best = 0
    for i in range(m):
        diag = 0
        for j in range(n):
            up = row[j + 1]
            row[j + 1] = 1 + min(up, row[j], diag) if matrix[i][j] == "1" else 0
            best = max(best, row[j + 1])
            diag = up
    return best * best


for matrix, expected in examples:
    assert maximal_square_rows(matrix) == expected


# Bài 5
def count_squares2(matrix):
    m, n = len(matrix), len(matrix[0])
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    total = 0
    for i in range(m):
        for j in range(n):
            if matrix[i][j] == "1":
                dp[i + 1][j + 1] = 1 + min(dp[i][j + 1], dp[i + 1][j], dp[i][j])
                total += dp[i + 1][j + 1]
    return total


assert count_squares2(grid("0111", "1111", "0111")) == 15
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (DP một hàng)

```python
class Solution:
    def maximalSquare(self, matrix: List[List[str]]) -> int:
        m, n = len(matrix), len(matrix[0])
        row = [0] * (n + 1)
        best = 0
        for i in range(m):
            diag = 0
            for j in range(n):
                up = row[j + 1]
                if matrix[i][j] == "1":
                    row[j + 1] = 1 + min(up, row[j], diag)
                    best = max(best, row[j + 1])
                else:
                    row[j + 1] = 0
                diag = up
        return best * best
```

**Checklist**

- [ ] `dp[i][j]` = **cạnh** hình vuông lớn nhất có góc **dưới-phải** tại `(i, j)`; công thức `1 + min(trên, trái, chéo)` cho ô `'1'`, `0` cho ô `'0'`?
- [ ] So sánh với **ký tự** `"1"` (không phải số `1`)?
- [ ] Có **hàng/cột đệm** `0` để khỏi lỗi chỉ số `-1`?
- [ ] Tạo `dp` bằng list comprehension (không dùng `[[0]*n]*m`)?
- [ ] Trả **diện tích** `best * best`?
- [ ] DP một hàng: `diag` được **reset `0`** mỗi hàng và cập nhật `diag = up` cuối mỗi bước?
- [ ] Đã thử toàn `0`, toàn `1`, một hàng/cột, vòng `1` có lỗ ở giữa, ca `[["0","1"],["1","1"]]`?
- [ ] Đã đối chiếu với brute force độc lập trên ma trận ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def maximalSquare(self, matrix: List[List[str]]) -> int`?